# Part 2 of analysis


First we read the data from the `R` section. 

In [1]:
# Import packages required
import scipy.io, scipy.sparse as sp
import anndata as ad, pandas as pd

In [ ]:
m = scipy.io.mmread("cnmf_input/counts.mtx")
X = sp.csr_matrix(m).T.tocsr()
genes = [l.strip() for l in open("cnmf_input/genes.txt")]
cells = [l.strip() for l in open("cnmf_input/barcodes.txt")]
adata = ad.AnnData(X=X,
                   obs = pd.DataFrame(index = cells),
                   var = pd.DataFrame(index = genes),
                   )
adata.write("cnmf_input/counts.h5ad")

/tmp/ipykernel_45391/2948331793.py:1: DeprecationWarning: The default value for `spmatrix` is changing to `False` in v1.20.
             That means the default return type will be a sparse array.
             Unless you use * instead of @, ** for matrix power, or you depend
             on 2D shapes from e.g. `A.sum(axis=0)` it may not matter to you.
             See the spmatrix to sparray migration guide for details.
             https://docs.scipy.org/doc/scipy/reference/sparse.migration_to_sparray.html
             
  m = scipy.io.mmread("cnmf_input/counts.mtx")


We are going to use `cnmf` to identify specific programs in gene expressing. 

In [1]:
from cnmf import cNMF
cnmf_obj = cNMF(output_dir="./cnmf_out", name="cnmf02") # This directory has to be relative 
# K and density_threshold must match the consensus run above (0.10 -> here 0.1)
usage, spectra_scores, spectra_tpm, top_genes = \
cnmf_obj.load_results(K=18, density_threshold=0.10)
usage = usage.div(usage.sum(axis=1), axis=0) # normalize per cell
usage.columns = [f"GEP{c}" for c in usage.columns] # tidy names
usage.to_csv("cnmf_out/usage_GEP.csv") # cells x programs
top_genes.head(100) # top genes per program

,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,MBNL1,JCHAIN,AOAH,TNFAIP3,TEX14,RPS18,PLVAP,APOA4,EPHB2,NALF1,HLA-DPB1,KRT19,NOTCH3,PDGFRA,BMX,HHIP,RIMS2,LYVE1
1,TTC39C,MZB1,ITGAE,CD69,CROCC,RPL41,FLT1,LCT,LGR5,LOC124901051,HLA-DRA,GPRC5A,CSRP2,ADAMTSL3,ITPRID1,NPNT,PCSK1N,MMRN1
2,ARHGAP15,TXNDC5,ATP8B4,KLF6,IER2,RPL10,KDR,PRAP1,RGMB,TLL1,HLA-DRB1,TSPAN8,RCAN2,COL6A1,IRAG2,DES,RGS7,PROX1
3,TC2N,IGHA1,CLNK,BTG1,ENSG00000285646,RPS14,EMCN,APOC3,OLFM4,RAB3C,HLA-DPA1,KRT20,PDE3A,COL3A1,PSTPIP2,DTNA,KCNH7,TFPI
4,PIP4K2A,SSR4,PRKCH,FOS,JUN,RPS12,NFIB,APOA1,MT-ATP8,ZNF385D,HLA-DQB1,RHPN2,ADRA1A,BICC1,ENSG00000231698,ACTG2,SCG5,RELN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,TAB2,CEP128,QTMAN,GPR183,NOP2,SNHG29,SHE,CTSE,ARHGEF10L,HMCN1,FTH1,CA7,MAP1B,NRG1,LOC105369668,GUSBP5,C12orf75,NRXN3
96,MYCBP2,LINC00571,PRKACB,ENSG00000231128,CLIC4,ACTG1,ENSG00000260971,SLC2A2,B3GAT1-DT,MMP28,RNF130,CDHR2,KCNAB1,INHBA,ENSG00000285695,UNC5C,CHGA,LOC105377329
97,STK10,CPEB4,ABR,PTMA,TMX4,TMSB10,CCN3,CXADR,SLC44A3,INMT,PHACTR1,HHLA2,AMPH,MIR100HG,CSMD1,PRDM6,TMEM61,LRCH2
98,CD6,ENSG00000257221,MYO9B,ZNF831,ENSG00000287978,RPL36AL,SORBS2,GATM,PIGR,DOC2B,HLA-DQA1.1,TJP1,MYLK,SLC25A48,FAT3,AKAP6,ONECUT3,GLYATL1-AS1


In [2]:
usage, spectra_scores, spectra_tpm, top_genes = cnmf_obj.load_results(K=18,density_threshold=0.10)

In [3]:
barrier_genes = ["TJP1", "OCLN", "CLDN1", "CLDN2", "CLDN3", "CLDN4", "CLDN7", "CDH1", "CTNNB1", "MUC2", "TFF3", "F11R"] 
genes_of_interest = barrier_genes + ["WNT5A"]

In [4]:
available = [g for g in genes_of_interest if g in spectra_scores.index]
missing = [g for g in genes_of_interest if g not in spectra_scores.index]
print("Not found in matrix: ", missing)

Not found in matrix:  []


In [5]:
gene_scores = spectra_scores.loc[available]
gene_scores.columns = [f'GEP{c}' for c in gene_scores.columns]
print(gene_scores)

            GEP1      GEP2      GEP3      GEP4      GEP5      GEP6      GEP7  \
TJP1   -0.000851 -0.000557 -0.000514 -0.000351 -0.000258 -0.000675  0.001306   
OCLN   -0.000275 -0.000192 -0.000145 -0.000138 -0.000148 -0.000598 -0.000040   
CLDN1  -0.000125 -0.000088 -0.000061 -0.000027 -0.000028 -0.000025 -0.000040   
CLDN2  -0.000101 -0.000004 -0.000057 -0.000029  0.000033 -0.000035 -0.000032   
CLDN3  -0.000670 -0.000457 -0.000436 -0.000311  0.000300  0.001004 -0.000202   
CLDN4  -0.000527 -0.000470 -0.000311 -0.000166  0.000144 -0.000027 -0.000158   
CLDN7  -0.000626 -0.000484 -0.000330 -0.000264 -0.000080 -0.000091 -0.000197   
CDH1   -0.000523 -0.000262 -0.000350 -0.000200 -0.000070 -0.000930 -0.000189   
CTNNB1  0.000190 -0.000798  0.000096  0.000264  0.000286 -0.001774  0.001100   
MUC2   -0.000041 -0.000041 -0.000015 -0.000046  0.000017 -0.000105 -0.000013   
TFF3   -0.000260 -0.000245 -0.000142 -0.000171  0.000105  0.000513 -0.000049   
F11R   -0.000286 -0.000222 -0.000230 -0.

In [6]:
gene_scores = spectra_tpm.loc[available]
gene_scores.columns = [f'GEP{c}' for c in gene_scores.columns]
print(gene_scores)

              GEP1      GEP2        GEP3        GEP4        GEP5       GEP6  \
TJP1      0.000000  0.000000    0.000000    0.000000    0.000000   0.000000   
OCLN      0.736023  0.000000    2.833322    0.000000    0.000000   0.000000   
CLDN1     0.000000  0.000000    0.000000    0.058849    0.000000   0.000000   
CLDN2     0.000000  1.107419    0.000389    0.296739    0.888540   0.000000   
CLDN3     0.000000  0.000000    0.000000    0.000000   26.224422  21.537930   
CLDN4     0.000000  0.000000    0.000000    0.000000    6.686691   0.000000   
CLDN7     0.000000  0.000000    0.000000    0.000000    0.000000  17.650145   
CDH1      0.000000  2.554365    0.000000    0.000000   15.154953   0.000000   
CTNNB1  195.104700  0.000000  184.195560  204.845460  203.002060   0.000000   
MUC2      0.000000  0.000000    0.000000    0.000000    0.000000   0.000000   
TFF3      0.000000  0.000000    0.000000    0.000000    0.000000  14.361702   
F11R     12.119829  7.914564    6.164822    0.000000